# 🐍 Day 19 · Python Basics — SQL Queries & Hiding Private Data (SQL Agent + PII)

> **Goal of this notebook:** The SQL agent writes SQL for you, and middleware hides private data (PII) such as emails and card numbers. Learn basic SQL and regex masking.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

✅ No API key and no internet needed for this notebook.

### 📋 What you will learn
1. Create a database in memory
2. Insert many rows with `executemany`
3. `SELECT ... WHERE` with safe placeholders
4. Why you must never build SQL with f-strings
5. Allow only read-only queries
6. Masking emails and card numbers with regex

## 1. A database in memory
`":memory:"` creates a temporary database that disappears when the program ends.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute('''CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    card TEXT,
    balance REAL
)''')
print("Table created ✅")

## 2. Insert many rows at once

In [ ]:
people = [
    ("Alice Smith", "alice@example.com", "4111 1111 1111 1111", 2500.0),
    ("Bob Kumar",   "bob@example.com",   "5500 0000 0000 0004", 120.5),
    ("Chen Li",     "chen@example.com",  "3400 0000 0000 009",  9800.0),
]
db.executemany("INSERT INTO users (name, email, card, balance) VALUES (?, ?, ?, ?)", people)
print(db.execute("SELECT COUNT(*) FROM users").fetchone()[0], "users added")

## 3. `SELECT ... WHERE` with placeholders
`?` is filled in safely by SQLite.

In [ ]:
name = "Alice Smith"
row = db.execute("SELECT name, balance FROM users WHERE name = ?", (name,)).fetchone()
print(row)

rich = db.execute("SELECT name FROM users WHERE balance > ? ORDER BY balance DESC", (1000,)).fetchall()
print(rich)

## 4. ⚠️ Never build SQL with f-strings
If a user types something tricky, an f-string query can be changed into a completely different query (this is called **SQL injection**).

In [ ]:
evil_input = "x' OR '1'='1"

dangerous = f"SELECT name FROM users WHERE name = '{evil_input}'"
print("The query becomes:", dangerous)
print("It returns EVERY user:", db.execute(dangerous).fetchall())

safe = db.execute("SELECT name FROM users WHERE name = ?", (evil_input,)).fetchall()
print("With a placeholder:", safe)       # nothing found — safe ✅

## 5. Allow only read-only queries
The SQL tool in today's lesson only accepts `SELECT` queries, so the LLM cannot delete data.

In [ ]:
def run_readonly(query):
    if not query.strip().lower().startswith("select"):
        return "❌ Only SELECT queries are allowed"
    return db.execute(query).fetchall()

print(run_readonly("SELECT name FROM users"))
print(run_readonly("DELETE FROM users"))

## 6. 🔗 Masking private data (PII)
`re.sub(pattern, replacement, text)` replaces every match. `PIIMiddleware` does this automatically before data reaches the LLM.

In [ ]:
import re

def redact(text):
    text = re.sub(r"[\w.]+@[\w.]+\.\w+", "[EMAIL]", text)                  # emails
    text = re.sub(r"\b(?:\d[ -]?){13,16}\b", "[CARD]", text)              # 13-16 digit card numbers
    return text

row = db.execute("SELECT name, email, card FROM users WHERE id = 1").fetchone()
raw = f"{row[0]} - {row[1]} - {row[2]}"
print("Before:", raw)
print("After: ", redact(raw))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Select only the `email` of `"Bob Kumar"` using a `?` placeholder.
2. Count how many users have a balance under 1000.
3. Use `re.sub` to replace every digit in `"Call 98765"` with `*`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(db.execute("SELECT email FROM users WHERE name = ?", ("Bob Kumar",)).fetchone()[0])
print(db.execute("SELECT COUNT(*) FROM users WHERE balance < ?", (1000,)).fetchone()[0])
print(re.sub(r"\d", "*", "Call 98765"))
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| In-memory DB | `sqlite3.connect(':memory:')` |
| Insert many | `db.executemany(sql, rows)` |
| Safe query | `db.execute('... WHERE name = ?', (n,))` |
| One / all rows | `.fetchone(), .fetchall()` |
| Read-only check | `q.lower().startswith('select')` |
| Regex replace | `re.sub(pattern, '[EMAIL]', text)` |

## ➡️ What's next?
You now know the Python used in **Day_19_LangChain_Agents_RAG_SQL_Middleware**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀